# Fase 5: Escenarios criticos e hiperparametros

Se evalua la robustez del modelo supervisado de cobranzas y se analiza la sensibilidad de sus hiperparametros. La separacion entre entrenamiento y prueba se conserva por `factura_id`.


## 1. Configuracion inicial

Se definen rutas relativas al proyecto, dependencias y carpetas de salida.


In [1]:
from pathlib import Path
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "05_escenarios_criticos_hiperparametros":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "AGENTS.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "AGENTS.md").exists():
            PROJECT_ROOT = parent
            break

PHASE_PATH = PROJECT_ROOT / "05_escenarios_criticos_hiperparametros"
print("Proyecto:", PROJECT_ROOT)
print("Fase:", PHASE_PATH)


Proyecto: C:\Users\KevinUbe\Documents\Proyectos\proyecto_integrador\GRUPO_3
Fase: C:\Users\KevinUbe\Documents\Proyectos\proyecto_integrador\GRUPO_3\05_escenarios_criticos_hiperparametros


## 2. Funciones base de escenarios

Este bloque define rutas, carga de datos, validaciones y metricas comunes. Se ejecuta una vez y luego las siguientes celdas usan esas funciones.


In [2]:
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


PROJECT_DIR = PROJECT_ROOT
PHASE_DIR = PHASE_PATH
OUTPUT_DIR = PHASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"

DATA_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "features_ml_prepared.csv"
TRAIN_IDS_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "train_facturas_ids.csv"
TEST_IDS_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "test_facturas_ids.csv"
MODEL_ARTIFACT_PATH = PROJECT_DIR / "04_evaluacion_modelos_ia" / "outputs" / "best_model_artifact.joblib"

TARGET_COL = "target_mora"
ID_COL = "factura_id"
DATE_COL = "fecha_corte"
RANDOM_SEEDS = list(range(101, 111))


def ensure_dirs() -> None:
    for path in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, ARTIFACTS_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    required = [DATA_PATH, TRAIN_IDS_PATH, TEST_IDS_PATH, MODEL_ARTIFACT_PATH]
    missing = [str(p.relative_to(PROJECT_DIR)) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Faltan entradas requeridas: {missing}")

    df = pd.read_csv(DATA_PATH)
    train_ids = pd.read_csv(TRAIN_IDS_PATH)[ID_COL].astype(str)
    test_ids = pd.read_csv(TEST_IDS_PATH)[ID_COL].astype(str)
    artifact = joblib.load(MODEL_ARTIFACT_PATH)

    feature_cols = artifact["feature_cols"]
    missing_cols = [c for c in [ID_COL, TARGET_COL, *feature_cols] if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Columnas faltantes en dataset preparado: {missing_cols}")

    df[ID_COL] = df[ID_COL].astype(str)
    train_df = df[df[ID_COL].isin(set(train_ids))].copy()
    test_df = df[df[ID_COL].isin(set(test_ids))].copy()
    overlap = set(train_df[ID_COL]).intersection(set(test_df[ID_COL]))
    if overlap:
        raise ValueError("El split oficial mezcla facturas entre train y test.")
    if train_df.empty or test_df.empty:
        raise ValueError("El split oficial produjo train o test vacio.")

    return df, train_df, test_df, artifact


def predict_metrics(
    X: pd.DataFrame,
    y: pd.Series,
    artifact: dict,
    scenario: str,
    measure_latency: bool = True,
) -> dict:
    preprocessor = artifact["preprocessor"]
    model = artifact["model"]
    encoder = artifact["target_encoder"]
    y_true = encoder.transform(y.astype(str))
    class_labels = list(range(len(encoder.classes_)))
    start = time.perf_counter()
    X_prepared = preprocessor.transform(X[artifact["feature_cols"]])
    y_pred = model.predict(X_prepared)
    y_proba = model.predict_proba(X_prepared) if hasattr(model, "predict_proba") else None
    elapsed = time.perf_counter() - start if measure_latency else np.nan

    metrics = {
        "escenario": scenario,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "latencia_total_s": elapsed,
        "latencia_ms_por_reg": (elapsed / max(len(X), 1)) * 1000 if measure_latency else np.nan,
        "n_registros": len(X),
    }

    if "+90" in encoder.classes_:
        plus_90 = int(np.where(encoder.classes_ == "+90")[0][0])
        metrics["recall_+90"] = recall_score(
            y_true, y_pred, labels=[plus_90], average="macro", zero_division=0
        )
        metrics["f1_+90"] = f1_score(
            y_true, y_pred, labels=[plus_90], average="macro", zero_division=0
        )
    else:
        metrics["recall_+90"] = np.nan
        metrics["f1_+90"] = np.nan

    if y_proba is not None:
        try:
            metrics["auc_weighted"] = roc_auc_score(
                y_true, y_proba, labels=class_labels, multi_class="ovr", average="weighted"
            )
        except ValueError:
            metrics["auc_weighted"] = np.nan
    else:
        metrics["auc_weighted"] = np.nan
    return metrics


def export_classification_report(
    X: pd.DataFrame, y: pd.Series, artifact: dict, path: Path
) -> pd.DataFrame:
    preprocessor = artifact["preprocessor"]
    model = artifact["model"]
    encoder = artifact["target_encoder"]
    y_true = encoder.transform(y.astype(str))
    X_prepared = preprocessor.transform(X[artifact["feature_cols"]])
    y_pred = model.predict(X_prepared)
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(encoder.classes_))),
        target_names=list(encoder.classes_),
        output_dict=True,
        zero_division=0,
    )
    df_report = pd.DataFrame(report).T.reset_index().rename(columns={"index": "clase"})
    df_report.to_csv(path, index=False)
    return df_report


## 3. Carga y validacion de entradas

Se cargan el dataset preparado, los splits oficiales y el modelo de fase 4. La tabla confirma que el split mantiene la unidad `factura_id`.


In [3]:
ensure_dirs()
df, train_df, test_df, artifact = load_inputs()
feature_cols = artifact["feature_cols"]
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].astype(str)
X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COL].astype(str)

split_summary = pd.DataFrame([
    {"particion": "train", "filas": len(train_df), "facturas": train_df[ID_COL].nunique()},
    {"particion": "test", "filas": len(test_df), "facturas": test_df[ID_COL].nunique()},
])
display(split_summary)
print("Modelo base:", artifact["model_name"])
print("Features usadas:", len(feature_cols))


,particion,filas,facturas
0,train,15735,4270
1,test,3936,1068


Modelo base: Logistic Regression
Features usadas: 39


## 4. Baseline limpio

El baseline es la referencia para medir degradacion. Tambien se exporta el reporte por clase.


In [4]:
baseline = predict_metrics(X_test, y_test, artifact, "baseline_clean")
baseline_df = pd.DataFrame([baseline])
baseline_df.to_csv(TABLES_DIR / "baseline_metrics.csv", index=False)
baseline_report = export_classification_report(
    X_test, y_test, artifact, TABLES_DIR / "classification_report_baseline.csv"
)

display(baseline_df.T.rename(columns={0: "valor"}))
display(baseline_report)


,valor
escenario,baseline_clean
accuracy,0.567327
balanced_accuracy,0.600614
f1_macro,0.57
f1_weighted,0.560607
precision_macro,0.562215
recall_macro,0.600614
latencia_total_s,0.009668
latencia_ms_por_reg,0.002456
n_registros,3936


,clase,precision,recall,f1-score,support
0,+30,0.423197,0.539281,0.474239,751.000000
1,+60,0.523077,0.392415,0.448422,1213.000000
2,+90,0.685455,0.579554,0.628072,1301.000000
3,on_time,0.617131,0.891207,0.729268,671.000000
4,accuracy,0.567327,0.567327,0.567327,0.567327
5,macro avg,0.562215,0.600614,0.570000,3936.000000
6,weighted avg,0.573726,0.567327,0.560607,3936.000000


## 5. Preparacion de escenarios de datos

Se agrupan variables por comportamiento operativo para simular ruido general, ausencia de datos y outliers de monto.


In [5]:
def numeric_features(df: pd.DataFrame, feature_cols: list[str]) -> list[str]:
    return [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]


def continuous_features(df: pd.DataFrame, feature_cols: list[str]) -> list[str]:
    excluded_prefixes = ("sector_",)
    excluded = {
        "tiene_garantia",
        "tiene_disputa_activa",
        "tiene_promesa_activa",
        "sin_gestion_previa",
        "esta_vencida_al_corte",
        "cliente_nuevo",
        "ultimo_resultado_enc",
    }
    cols = []
    for col in numeric_features(df, feature_cols):
        if col in excluded or col.startswith(excluded_prefixes):
            continue
        if df[col].nunique(dropna=True) > 5:
            cols.append(col)
    return cols


def add_gaussian_noise(X: pd.DataFrame, cols: list[str], level: float, rng: np.random.Generator) -> pd.DataFrame:
    Xn = X.copy()
    for col in cols:
        sigma = Xn[col].std(skipna=True)
        if pd.notna(sigma) and sigma > 0:
            Xn[col] = Xn[col] + rng.normal(0, sigma * level, size=len(Xn))
            if X[col].min(skipna=True) >= 0:
                Xn[col] = Xn[col].clip(lower=0)
            if X[col].min(skipna=True) >= 0 and X[col].max(skipna=True) <= 1:
                Xn[col] = Xn[col].clip(lower=0, upper=1)
    return Xn


def add_random_missing(X: pd.DataFrame, cols: list[str], level: float, rng: np.random.Generator) -> pd.DataFrame:
    Xm = X.copy()
    mask = rng.random((len(Xm), len(cols))) < level
    for idx, col in enumerate(cols):
        Xm.loc[mask[:, idx], col] = np.nan
    return Xm


def add_directed_missing(X: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    Xm = X.copy()
    for col in cols:
        if col in Xm.columns:
            Xm[col] = np.nan
    return Xm


def add_outliers(
    X: pd.DataFrame,
    cols: list[str],
    multiplier: float,
    row_share: float,
    rng: np.random.Generator,
) -> pd.DataFrame:
    Xo = X.copy()
    n = max(1, int(len(Xo) * row_share))
    idx = rng.choice(Xo.index.to_numpy(), size=n, replace=False)
    for col in cols:
        if col in Xo.columns:
            Xo.loc[idx, col] = Xo.loc[idx, col] * multiplier
    return Xo


def build_noise_scenarios(X_test: pd.DataFrame, feature_cols: list[str]) -> list[dict]:
    cont_cols = continuous_features(X_test, feature_cols)
    gestion_cols = [
        "num_gestiones_factura",
        "dias_desde_ultima_gestion",
        "tasa_contacto_cliente",
        "ultimo_resultado_enc",
        "num_no_contesta_cons",
        "sin_gestion_previa",
        "intensidad_gestion",
        "friccion_contacto",
    ]
    promesa_cols = [
        "tiene_promesa_activa",
        "num_promesas_rotas",
        "tasa_cumpl_promesas",
        "promesas_total",
        "ratio_promesas_rotas",
    ]
    historial_cols = [
        "num_facturas_prev",
        "mora_promedio_hist",
        "mora_ultimo_tramo",
        "tasa_cumplimiento",
        "monto_promedio_hist",
        "moras_consecutivas",
    ]
    monto_cols = ["monto", "monto_promedio_hist", "ratio_monto"]

    scenarios = []
    for level in [0.05, 0.10, 0.20]:
        scenarios.append(
            {
                "name": f"ruido_gaussiano_{int(level * 100)}pct",
                "family": "ruido_gaussiano",
                "level": level,
                "builder": lambda X, rng, level=level: add_gaussian_noise(X, cont_cols, level, rng),
            }
        )
    for level in [0.05, 0.15, 0.30]:
        scenarios.append(
            {
                "name": f"missing_aleatorio_{int(level * 100)}pct",
                "family": "missing_aleatorio",
                "level": level,
                "builder": lambda X, rng, level=level: add_random_missing(X, cont_cols, level, rng),
            }
        )
    scenarios.extend(
        [
            {
                "name": "missing_dirigido_gestion",
                "family": "missing_dirigido",
                "level": 1.00,
                "builder": lambda X, rng: add_directed_missing(X, gestion_cols),
            },
            {
                "name": "missing_dirigido_promesas",
                "family": "missing_dirigido",
                "level": 1.00,
                "builder": lambda X, rng: add_directed_missing(X, promesa_cols),
            },
            {
                "name": "missing_dirigido_historial",
                "family": "missing_dirigido",
                "level": 1.00,
                "builder": lambda X, rng: add_directed_missing(X, historial_cols),
            },
        ]
    )
    for mult, share in [(3, 0.05), (5, 0.05), (10, 0.05), (5, 0.10)]:
        scenarios.append(
            {
                "name": f"outlier_monto_{mult}x_{int(share * 100)}pct",
                "family": "outlier_monto",
                "level": share,
                "builder": lambda X, rng, mult=mult, share=share: add_outliers(
                    X, monto_cols, mult, share, rng
                ),
            }
        )
    scenarios.append(
        {
            "name": "outlier_critico_combinado",
            "family": "combinado",
            "level": 1.00,
            "builder": lambda X, rng: add_gaussian_noise(
                add_random_missing(
                    add_outliers(X, monto_cols, 10, 0.05, rng), gestion_cols, 0.50, rng
                ),
                cont_cols,
                0.20,
                rng,
            ),
        }
    )
    return scenarios


In [6]:
continuous_cols = continuous_features(X_test, feature_cols)
scenario_catalog = pd.DataFrame([
    {"grupo": "continuas", "n_variables": len(continuous_cols), "uso": "ruido gaussiano y missing aleatorio"},
    {"grupo": "gestion", "n_variables": 8, "uso": "missing dirigido por fallas de gestion"},
    {"grupo": "promesas", "n_variables": 5, "uso": "missing dirigido por fallas en promesas"},
    {"grupo": "historial", "n_variables": 6, "uso": "missing dirigido por historial incompleto"},
    {"grupo": "monto", "n_variables": 3, "uso": "outliers de monto"},
])
display(scenario_catalog)


,grupo,n_variables,uso
0,continuas,23,ruido gaussiano y missing aleatorio
1,gestion,8,missing dirigido por fallas de gestion
2,promesas,5,missing dirigido por fallas en promesas
3,historial,6,missing dirigido por historial incompleto
4,monto,3,outliers de monto


## 6. Ejecucion de ruido, missing y outliers

Cada escenario se repite con diez semillas para obtener media, desviacion e intervalo de confianza.


In [7]:
def summarize_runs(run_df: pd.DataFrame, baseline_f1: float) -> pd.DataFrame:
    rows = []
    for scenario, group in run_df.groupby("escenario", sort=False):
        f1 = group["f1_macro"].astype(float)
        delta = baseline_f1 - f1
        sem = f1.sem(ddof=1) if len(f1) > 1 else 0.0
        try:
            p_value = stats.ttest_1samp(delta, 0.0, nan_policy="omit").pvalue
        except Exception:
            p_value = np.nan
        first = group.iloc[0]
        rows.append(
            {
                "escenario": scenario,
                "familia": first.get("familia", ""),
                "nivel": first.get("nivel", np.nan),
                "n_runs": len(group),
                "f1_macro_mean": f1.mean(),
                "f1_macro_std": f1.std(ddof=1),
                "f1_macro_ci95_low": f1.mean() - 1.96 * sem,
                "f1_macro_ci95_high": f1.mean() + 1.96 * sem,
                "balanced_accuracy_mean": group["balanced_accuracy"].mean(),
                "recall_+90_mean": group["recall_+90"].mean(),
                "degradacion_f1_macro_pct_mean": ((baseline_f1 - f1.mean()) / baseline_f1) * 100,
                "degradacion_f1_macro_pct_std": (delta / baseline_f1 * 100).std(ddof=1),
                "ttest_degradacion_pvalue": p_value,
            }
        )
    return pd.DataFrame(rows)


In [8]:
raw_rows = []
for scenario in build_noise_scenarios(X_test, feature_cols):
    for seed in RANDOM_SEEDS:
        rng = np.random.default_rng(seed)
        X_scenario = scenario["builder"](X_test, rng)
        row = predict_metrics(X_scenario, y_test, artifact, scenario["name"])
        row["familia"] = scenario["family"]
        row["nivel"] = scenario["level"]
        row["seed"] = seed
        raw_rows.append(row)

scenario_runs = pd.DataFrame(raw_rows)
scenario_runs["degradacion_f1_macro_pct"] = (
    baseline["f1_macro"] - scenario_runs["f1_macro"]
) / baseline["f1_macro"] * 100
scenario_runs.to_csv(TABLES_DIR / "scenario_run_metrics.csv", index=False)

noise_summary = summarize_runs(scenario_runs, baseline["f1_macro"])
noise_summary.to_csv(TABLES_DIR / "noise_missing_outlier_results.csv", index=False)
display(noise_summary.sort_values("degradacion_f1_macro_pct_mean", ascending=False).head(10))


C:\Users\KevinUbe\Documents\Proyectos\proyecto_integrador\GRUPO_3\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
C:\Users\KevinUbe\Documents\Proyectos\proyecto_integrador\GRUPO_3\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
C:\Users\KevinUbe\Documents\Proyectos\proyecto_integrador\GRUPO_3\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun

,escenario,familia,nivel,n_runs,f1_macro_mean,f1_macro_std,f1_macro_ci95_low,f1_macro_ci95_high,balanced_accuracy_mean,recall_+90_mean,degradacion_f1_macro_pct_mean,degradacion_f1_macro_pct_std,ttest_degradacion_pvalue
13,outlier_critico_combinado,combinado,1.00,10,0.418005,8.258797e-03,0.412886,0.423124,0.411480,0.701307,26.665850,1.448911e+00,6.575569e-13
5,missing_aleatorio_30pct,missing_aleatorio,0.30,10,0.443014,5.625098e-03,0.439527,0.446500,0.437617,0.441045,22.278324,9.868589e-01,1.049757e-13
2,ruido_gaussiano_20pct,ruido_gaussiano,0.20,10,0.485311,6.129748e-03,0.481511,0.489110,0.489062,0.684012,14.857837,1.075394e+00,8.610365e-12
6,missing_dirigido_gestion,missing_dirigido,1.00,10,0.490236,0.000000e+00,0.490236,0.490236,0.478525,0.535742,13.993751,0.000000e+00,3.779022e-143
4,missing_aleatorio_15pct,missing_aleatorio,0.15,10,0.507505,6.018274e-03,0.503775,0.511235,0.510345,0.507379,10.964115,1.055837e+00,1.108443e-10
1,ruido_gaussiano_10pct,ruido_gaussiano,0.10,10,0.523983,3.263192e-03,0.521960,0.526006,0.541281,0.663797,8.073200,5.724896e-01,7.167119e-12
0,ruido_gaussiano_5pct,ruido_gaussiano,0.05,10,0.543247,3.626929e-03,0.540999,0.545495,0.568316,0.643428,4.693548,6.363032e-01,2.329083e-09
3,missing_aleatorio_5pct,missing_aleatorio,0.05,10,0.547359,4.454721e-03,0.544598,0.550120,0.567477,0.555419,3.972114,7.815298e-01,6.184781e-08
8,missing_dirigido_historial,missing_dirigido,1.00,10,0.555063,0.000000e+00,0.555063,0.555063,0.578623,0.607994,2.620541,4.681111e-16,0.000000e+00
7,missing_dirigido_promesas,missing_dirigido,1.00,10,0.562474,1.170278e-16,0.562474,0.562474,0.596634,0.531130,1.320300,2.340556e-16,0.000000e+00


## 7. Drift formal

PSI cuantifica magnitud del cambio de distribucion. KS contrasta train contra test por variable.


In [9]:
def calculate_psi(reference: np.ndarray, evaluation: np.ndarray, bins: int = 10) -> float:
    ref = pd.Series(reference).replace([np.inf, -np.inf], np.nan).dropna()
    eva = pd.Series(evaluation).replace([np.inf, -np.inf], np.nan).dropna()
    if ref.nunique() < 2 or eva.nunique() < 2:
        return 0.0
    edges = np.unique(np.nanquantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0] = -np.inf
    edges[-1] = np.inf
    ref_pct = np.histogram(ref, bins=edges)[0] / len(ref)
    eva_pct = np.histogram(eva, bins=edges)[0] / len(eva)
    eps = 1e-6
    ref_pct = np.clip(ref_pct, eps, None)
    eva_pct = np.clip(eva_pct, eps, None)
    return float(np.sum((eva_pct - ref_pct) * np.log(eva_pct / ref_pct)))


def drift_statistics(train_df: pd.DataFrame, eval_df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in numeric_features(train_df, feature_cols):
        ref = train_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        eva = eval_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(ref) < 2 or len(eva) < 2:
            continue
        ks_stat, ks_p = stats.ks_2samp(ref, eva)
        rows.append(
            {
                "variable": col,
                "psi": calculate_psi(ref.to_numpy(), eva.to_numpy()),
                "ks_statistic": float(ks_stat),
                "ks_pvalue": float(ks_p),
                "media_train": float(ref.mean()),
                "media_eval": float(eva.mean()),
                "diff_media_abs": float(eva.mean() - ref.mean()),
            }
        )
    return pd.DataFrame(rows).sort_values(["psi", "ks_statistic"], ascending=False)


In [10]:
psi_ks = drift_statistics(train_df, test_df, feature_cols)
psi_ks.to_csv(TABLES_DIR / "drift_psi_ks.csv", index=False)
display(psi_ks.head(12))


,variable,psi,ks_statistic,ks_pvalue,media_train,media_eval,diff_media_abs
2,ratio_monto,0.031954,0.044845,6.124139e-06,1.086747,1.094122,0.007374
13,antiguedad_meses,0.030854,0.028520,1.170590e-02,36.412266,36.582317,0.170051
20,tasa_cumpl_promesas,0.023102,0.065427,3.688504e-12,0.606110,0.579662,-0.026448
23,ratio_promesas_rotas,0.023083,0.061440,8.987762e-11,0.360754,0.389444,0.028690
4,mora_ultimo_tramo,0.022565,0.035422,7.231998e-04,20.333128,20.970361,0.637233
1,monto_promedio_hist,0.019171,0.045518,4.171685e-06,27377.399884,27734.477975,357.078091
0,monto,0.018316,0.039446,1.081156e-04,28084.273444,28529.875620,445.602176
9,num_promesas_rotas,0.017896,0.051098,1.391024e-07,2.671878,2.905996,0.234118
3,mora_promedio_hist,0.017176,0.043440,1.339360e-05,23.707725,24.790267,1.082541
14,num_facturas_prev,0.016795,0.043793,1.103078e-05,16.002542,16.752795,0.750253


## 8. Drift temporal, sectorial y de cartera critica

Se mide desempeno sobre subconjuntos de interes: futuro temporal, sectores y facturas con mayor mora historica.


In [11]:
drift_rows = []
if DATE_COL in df.columns:
    temp = df.copy()
    temp[DATE_COL] = pd.to_datetime(temp[DATE_COL], errors="coerce")
    invoice_dates = temp.groupby(ID_COL)[DATE_COL].max().dropna().sort_values()
    cutoff = invoice_dates.quantile(0.70)
    temporal_train_ids = set(invoice_dates[invoice_dates <= cutoff].index.astype(str))
    temporal_test_ids = set(invoice_dates[invoice_dates > cutoff].index.astype(str))
    temp_future = temp[temp[ID_COL].isin(temporal_test_ids)].copy()
    temp_past = temp[temp[ID_COL].isin(temporal_train_ids)].copy()
    if not temp_future.empty:
        metric = predict_metrics(
            temp_future[feature_cols],
            temp_future[TARGET_COL].astype(str),
            artifact,
            "drift_temporal_futuro",
        )
        metric["tipo_drift"] = "temporal"
        drift_rows.append(metric)
    pd.DataFrame([{
        "fecha_col": DATE_COL,
        "fecha_corte_temporal": cutoff.date().isoformat(),
        "n_facturas_train_temporal": len(temporal_train_ids),
        "n_facturas_test_temporal": len(temporal_test_ids),
        "n_registros_train_temporal": len(temp_past),
        "n_registros_test_temporal": len(temp_future),
        "interseccion_factura_id": len(temporal_train_ids.intersection(temporal_test_ids)),
        "fecha_max_train": temp_past[DATE_COL].max().date().isoformat(),
        "fecha_min_test": temp_future[DATE_COL].min().date().isoformat(),
    }]).to_csv(TABLES_DIR / "temporal_drift_split_summary.csv", index=False)

sector_rows = []
for col in [c for c in feature_cols if c.startswith("sector_")]:
    subset_mask = X_test[col] == 1
    if subset_mask.sum() < 30:
        continue
    sector_name = col.replace("sector_", "")
    metric = predict_metrics(X_test[subset_mask], y_test[subset_mask], artifact, f"drift_sector_{sector_name}")
    metric["sector"] = sector_name
    metric["tipo_drift"] = "covariate_sector"
    drift_rows.append(metric)
    sector_rows.append({
        "sector": sector_name,
        "n_registros": int(subset_mask.sum()),
        "f1_macro": metric["f1_macro"],
        "recall_+90": metric["recall_+90"],
        "degradacion_f1_macro_pct": (baseline["f1_macro"] - metric["f1_macro"]) / baseline["f1_macro"] * 100,
    })

if "mora_promedio_hist" in X_test.columns:
    threshold = X_train["mora_promedio_hist"].quantile(0.75)
    mask = X_test["mora_promedio_hist"] >= threshold
    if mask.sum() >= 30:
        metric = predict_metrics(X_test[mask], y_test[mask], artifact, "drift_cartera_critica_q75")
        metric["tipo_drift"] = "covariate_riesgo"
        drift_rows.append(metric)

drift_results = pd.DataFrame(drift_rows)
drift_results["degradacion_f1_macro_pct"] = (
    baseline["f1_macro"] - drift_results["f1_macro"]
) / baseline["f1_macro"] * 100
drift_results.to_csv(TABLES_DIR / "drift_results.csv", index=False)
pd.DataFrame(sector_rows).sort_values("degradacion_f1_macro_pct", ascending=False).to_csv(
    TABLES_DIR / "drift_sector_results.csv", index=False
)
display(drift_results.sort_values("degradacion_f1_macro_pct", ascending=False))


,escenario,accuracy,balanced_accuracy,f1_macro,f1_weighted,precision_macro,recall_macro,latencia_total_s,latencia_ms_por_reg,n_registros,recall_+90,f1_+90,auc_weighted,tipo_drift,sector,degradacion_f1_macro_pct
9,drift_cartera_critica_q75,0.528050,0.406236,0.378500,0.528203,0.373656,0.406236,0.004714,0.004198,1123,0.619485,0.647454,0.722017,covariate_riesgo,NaN,33.596592
1,drift_sector_retail,0.548913,0.583976,0.535259,0.540546,0.547418,0.583976,0.004335,0.011780,368,0.565217,0.464286,0.819891,covariate_sector,retail,6.095026
6,drift_sector_tecnologia,0.572785,0.565029,0.542946,0.566513,0.539969,0.565029,0.003623,0.011466,316,0.635593,0.694444,0.842457,covariate_sector,tecnologia,4.746281
2,drift_sector_manufactura,0.554953,0.591411,0.547847,0.532647,0.541979,0.591411,0.004325,0.005869,737,0.677043,0.644444,0.820472,covariate_sector,manufactura,3.886473
4,drift_sector_construccion,0.549563,0.587347,0.550969,0.554559,0.544771,0.587347,0.005502,0.008021,686,0.556452,0.634483,0.787614,covariate_sector,construccion,3.338816
5,drift_sector_agro,0.574032,0.591304,0.559769,0.566658,0.551711,0.591304,0.004270,0.009727,439,0.560976,0.625850,0.826467,covariate_sector,agro,1.794884
0,drift_temporal_futuro,0.559685,0.612248,0.564656,0.560639,0.551244,0.612248,0.009062,0.001321,6861,0.580543,0.652277,0.805667,temporal,NaN,0.937612
7,drift_sector_salud,0.577143,0.600392,0.579256,0.570019,0.607265,0.600392,0.003748,0.007139,525,0.445255,0.578199,0.811204,covariate_sector,salud,-1.623856
8,drift_sector_transporte,0.580645,0.625030,0.580960,0.577182,0.561170,0.625030,0.003909,0.006637,589,0.588235,0.646552,0.799060,covariate_sector,transporte,-1.922771
3,drift_sector_servicios,0.605072,0.649269,0.622234,0.598732,0.626166,0.649269,0.003574,0.012950,276,0.500000,0.589147,0.804793,covariate_sector,servicios,-9.163894


## 9. Pruebas de estres

Se revisan condiciones limite de entrada, esquema, volumen, latencia y concurrencia.


In [12]:
def run_stress_tests(X_test: pd.DataFrame, y_test: pd.Series, artifact: dict, baseline: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    X_extreme = X_test.copy()
    assignments = {
        "monto": X_test["monto"].quantile(0.99) * 25 if "monto" in X_test else np.nan,
        "ratio_monto": X_test["ratio_monto"].quantile(0.99) * 15 if "ratio_monto" in X_test else np.nan,
        "dias_hasta_vence": -365,
        "dias_hasta_vence_pos": 0,
        "tasa_contacto_cliente": 1.8,
        "tasa_cumplimiento": -0.25,
        "num_no_contesta_cons": X_test.get("num_no_contesta_cons", pd.Series([0])).max() + 20,
    }
    for col, value in assignments.items():
        if col in X_extreme.columns and pd.notna(value):
            X_extreme[col] = value
    rows.append(
        {
            **predict_metrics(X_extreme, y_test, artifact, "stress_inputs_extremos"),
            "tipo_stress": "inputs_extremos",
            "condicion_limite": "montos altos, tasas fuera de rango y dias negativos",
            "resultado_tecnico": "prediccion_completa",
        }
    )

    X_extra = X_test.copy()
    X_extra["columna_irrelevante_no_schema"] = 1
    rows.append(
        {
            **predict_metrics(X_extra, y_test, artifact, "stress_columna_extra"),
            "tipo_stress": "schema",
            "condicion_limite": "columna adicional no esperada",
            "resultado_tecnico": "ignorada_por_column_transformer",
        }
    )

    missing_col_row = {
        "escenario": "stress_columna_requerida_faltante",
        "tipo_stress": "schema",
        "condicion_limite": "se elimina una feature requerida antes de inferencia",
        "resultado_tecnico": "falla_controlada",
        "n_registros": len(X_test),
    }
    try:
        feature_to_drop = artifact["feature_cols"][0]
        predict_metrics(X_test.drop(columns=[feature_to_drop]), y_test, artifact, "stress_columna_requerida_faltante")
        missing_col_row["resultado_tecnico"] = "no_fallo"
    except Exception as exc:
        missing_col_row["error"] = type(exc).__name__
    rows.append(missing_col_row)

    latency_rows = []
    for multiplier in [1, 10, 50, 100]:
        X_vol = pd.concat([X_test] * multiplier, ignore_index=True)
        y_vol = pd.concat([y_test] * multiplier, ignore_index=True)
        metric = predict_metrics(X_vol, y_vol, artifact, f"stress_volumen_{multiplier}x")
        metric["tipo_stress"] = "volumen"
        metric["condicion_limite"] = f"{len(X_vol)} registros en lote"
        metric["resultado_tecnico"] = "prediccion_completa"
        rows.append(metric)

    for batch_size in [1, 10, 100, 1000, len(X_test)]:
        sample = X_test.iloc[: min(batch_size, len(X_test))].copy()
        y_sample = y_test.iloc[: len(sample)].copy()
        metric = predict_metrics(sample, y_sample, artifact, f"latencia_lote_{len(sample)}")
        latency_rows.append(
            {
                "escenario": metric["escenario"],
                "tamano_lote": len(sample),
                "latencia_total_s": metric["latencia_total_s"],
                "latencia_ms_por_reg": metric["latencia_ms_por_reg"],
                "throughput_reg_s": len(sample) / max(metric["latencia_total_s"], 1e-9),
            }
        )

    def predict_small_batch(_: int) -> float:
        sample = X_test.iloc[: min(250, len(X_test))].copy()
        start = time.perf_counter()
        artifact["model"].predict(artifact["preprocessor"].transform(sample[artifact["feature_cols"]]))
        return time.perf_counter() - start

    for workers in [1, 2, 4, 8]:
        start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=workers) as executor:
            durations = list(executor.map(predict_small_batch, range(workers * 4)))
        elapsed = time.perf_counter() - start
        latency_rows.append(
            {
                "escenario": f"concurrencia_{workers}_workers",
                "tamano_lote": min(250, len(X_test)),
                "workers": workers,
                "corridas": workers * 4,
                "latencia_total_s": elapsed,
                "latencia_ms_por_reg": (elapsed / (min(250, len(X_test)) * workers * 4)) * 1000,
                "p95_duracion_batch_s": float(np.percentile(durations, 95)),
                "throughput_reg_s": (min(250, len(X_test)) * workers * 4) / max(elapsed, 1e-9),
            }
        )

    stress_df = pd.DataFrame(rows)
    stress_df["degradacion_f1_macro_pct"] = np.where(
        stress_df["f1_macro"].notna(),
        (baseline["f1_macro"] - stress_df["f1_macro"]) / baseline["f1_macro"] * 100,
        np.nan,
    )
    return stress_df, pd.DataFrame(latency_rows)


In [13]:
stress_results, latency_results = run_stress_tests(X_test, y_test, artifact, baseline)
stress_results.to_csv(TABLES_DIR / "stress_test_results.csv", index=False)
latency_results.to_csv(TABLES_DIR / "latency_concurrency_results.csv", index=False)
display(stress_results[["escenario", "tipo_stress", "f1_macro", "degradacion_f1_macro_pct", "resultado_tecnico"]])
display(latency_results)


,escenario,tipo_stress,f1_macro,degradacion_f1_macro_pct,resultado_tecnico
0,stress_inputs_extremos,inputs_extremos,0.124212,78.20837,prediccion_completa
1,stress_columna_extra,schema,0.570000,0.00000,ignorada_por_column_transformer
2,stress_columna_requerida_faltante,schema,NaN,NaN,falla_controlada
3,stress_volumen_1x,volumen,0.570000,0.00000,prediccion_completa
4,stress_volumen_10x,volumen,0.570000,0.00000,prediccion_completa
5,stress_volumen_50x,volumen,0.570000,0.00000,prediccion_completa
6,stress_volumen_100x,volumen,0.570000,0.00000,prediccion_completa


,escenario,tamano_lote,latencia_total_s,latencia_ms_por_reg,throughput_reg_s,workers,corridas,p95_duracion_batch_s
0,latencia_lote_1,1,0.004608,4.607900,217.018596,NaN,NaN,NaN
1,latencia_lote_10,10,0.003336,0.333580,2997.781622,NaN,NaN,NaN
2,latencia_lote_100,100,0.003959,0.039587,25260.817254,NaN,NaN,NaN
3,latencia_lote_1000,1000,0.003961,0.003961,252467.872910,NaN,NaN,NaN
4,latencia_lote_3936,3936,0.007042,0.001789,558924.188154,NaN,NaN,NaN
5,concurrencia_1_workers,250,0.015948,0.015948,62705.360008,1.0,4.0,0.003864
6,concurrencia_2_workers,250,0.033582,0.016791,59555.182229,2.0,8.0,0.014247
7,concurrencia_4_workers,250,0.068201,0.017050,58650.166431,4.0,16.0,0.022210
8,concurrencia_8_workers,250,0.134766,0.016846,59362.285817,8.0,32.0,0.053570


## 10. Visualizaciones de robustez

Los graficos permiten revisar rapidamente que escenarios degradan mas el desempeno y donde aparece drift.


In [14]:
def make_figures(
    noise_summary: pd.DataFrame,
    drift_results: pd.DataFrame,
    psi_ks: pd.DataFrame,
    latency_df: pd.DataFrame,
    all_summary: pd.DataFrame,
) -> None:
    plt.style.use("default")

    plot_df = noise_summary[noise_summary["familia"].isin(["ruido_gaussiano", "missing_aleatorio", "outlier_monto"])]
    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        for family, group in plot_df.groupby("familia"):
            ax.errorbar(
                group["escenario"],
                group["f1_macro_mean"],
                yerr=group["f1_macro_std"].fillna(0),
                marker="o",
                label=family,
            )
        ax.set_title("Sensibilidad ante ruido, missing y outliers")
        ax.set_ylabel("F1-macro promedio")
        ax.tick_params(axis="x", rotation=35)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "noise_missing_outlier_impact.png", dpi=150)
        plt.close(fig)

    if not drift_results.empty:
        fig, ax = plt.subplots(figsize=(9, 4))
        drift_results.sort_values("degradacion_f1_macro_pct").plot(
            x="escenario", y="degradacion_f1_macro_pct", kind="barh", ax=ax, legend=False
        )
        ax.set_title("Impacto de escenarios de drift")
        ax.set_xlabel("Degradacion F1-macro (%)")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "drift_impact.png", dpi=150)
        plt.close(fig)

    if not psi_ks.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        top = psi_ks.head(12).iloc[::-1]
        ax.barh(top["variable"], top["psi"])
        ax.axvline(0.10, color="orange", linestyle="--", linewidth=1, label="PSI 0.10")
        ax.axvline(0.25, color="red", linestyle="--", linewidth=1, label="PSI 0.25")
        ax.set_title("Variables con mayor drift PSI")
        ax.set_xlabel("PSI")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "drift_psi_top_variables.png", dpi=150)
        plt.close(fig)

    if not latency_df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        subset = latency_df[latency_df["escenario"].str.startswith("latencia_lote")].copy()
        ax.plot(subset["tamano_lote"], subset["latencia_ms_por_reg"], marker="o")
        ax.set_xscale("log")
        ax.set_title("Latencia por tamano de lote")
        ax.set_xlabel("Registros")
        ax.set_ylabel("ms por registro")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "latency_by_batch_size.png", dpi=150)
        plt.close(fig)

    if not all_summary.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        top = all_summary.sort_values("degradacion_f1_macro_pct", ascending=False).head(15).iloc[::-1]
        ax.barh(top["escenario"], top["degradacion_f1_macro_pct"])
        ax.set_title("Escenarios con mayor degradacion")
        ax.set_xlabel("Degradacion F1-macro (%)")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "top_degradation_scenarios.png", dpi=150)
        plt.close(fig)


In [15]:
all_scenarios = pd.concat(
    [
        baseline_df.assign(familia="baseline", degradacion_f1_macro_pct=np.nan),
        noise_summary.rename(
            columns={
                "f1_macro_mean": "f1_macro",
                "balanced_accuracy_mean": "balanced_accuracy",
                "recall_+90_mean": "recall_+90",
            }
        ),
        drift_results,
        stress_results,
    ],
    ignore_index=True,
    sort=False,
)
all_scenarios.to_csv(TABLES_DIR / "all_scenarios_summary.csv", index=False)
make_figures(noise_summary, drift_results, psi_ks, latency_results, all_scenarios)
display(all_scenarios.sort_values("degradacion_f1_macro_pct", ascending=False).head(12))


,escenario,accuracy,balanced_accuracy,f1_macro,f1_weighted,precision_macro,recall_macro,latencia_total_s,latencia_ms_por_reg,n_registros,...,f1_macro_ci95_high,degradacion_f1_macro_pct_mean,degradacion_f1_macro_pct_std,ttest_degradacion_pvalue,tipo_drift,sector,tipo_stress,condicion_limite,resultado_tecnico,error
25,stress_inputs_extremos,0.330539,0.250000,0.124212,0.164228,0.082635,0.250000,0.013993,0.003555,3936.0,...,NaN,NaN,NaN,NaN,NaN,NaN,inputs_extremos,"montos altos, tasas fuera de rango y dias nega...",prediccion_completa,NaN
24,drift_cartera_critica_q75,0.528050,0.406236,0.378500,0.528203,0.373656,0.406236,0.004714,0.004198,1123.0,...,NaN,NaN,NaN,NaN,covariate_riesgo,NaN,NaN,NaN,NaN,NaN
16,drift_sector_retail,0.548913,0.583976,0.535259,0.540546,0.547418,0.583976,0.004335,0.011780,368.0,...,NaN,NaN,NaN,NaN,covariate_sector,retail,NaN,NaN,NaN,NaN
21,drift_sector_tecnologia,0.572785,0.565029,0.542946,0.566513,0.539969,0.565029,0.003623,0.011466,316.0,...,NaN,NaN,NaN,NaN,covariate_sector,tecnologia,NaN,NaN,NaN,NaN
17,drift_sector_manufactura,0.554953,0.591411,0.547847,0.532647,0.541979,0.591411,0.004325,0.005869,737.0,...,NaN,NaN,NaN,NaN,covariate_sector,manufactura,NaN,NaN,NaN,NaN
19,drift_sector_construccion,0.549563,0.587347,0.550969,0.554559,0.544771,0.587347,0.005502,0.008021,686.0,...,NaN,NaN,NaN,NaN,covariate_sector,construccion,NaN,NaN,NaN,NaN
20,drift_sector_agro,0.574032,0.591304,0.559769,0.566658,0.551711,0.591304,0.004270,0.009727,439.0,...,NaN,NaN,NaN,NaN,covariate_sector,agro,NaN,NaN,NaN,NaN
15,drift_temporal_futuro,0.559685,0.612248,0.564656,0.560639,0.551244,0.612248,0.009062,0.001321,6861.0,...,NaN,NaN,NaN,NaN,temporal,NaN,NaN,NaN,NaN,NaN
30,stress_volumen_50x,0.567327,0.600614,0.570000,0.560607,0.562215,0.600614,0.212926,0.001082,196800.0,...,NaN,NaN,NaN,NaN,NaN,NaN,volumen,196800 registros en lote,prediccion_completa,NaN
29,stress_volumen_10x,0.567327,0.600614,0.570000,0.560607,0.562215,0.600614,0.042106,0.001070,39360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,volumen,39360 registros en lote,prediccion_completa,NaN


## 11. Funciones de tuning

La busqueda se realiza sobre Logistic Regression. El test externo se conserva solo para evaluacion final.


In [16]:

import itertools
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


PROJECT_DIR = PROJECT_ROOT
PHASE_DIR = PHASE_PATH
OUTPUT_DIR = PHASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"

DATA_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "features_ml_prepared.csv"
TRAIN_IDS_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "train_facturas_ids.csv"
TEST_IDS_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "test_facturas_ids.csv"
MODEL_ARTIFACT_PATH = PROJECT_DIR / "04_evaluacion_modelos_ia" / "outputs" / "best_model_artifact.joblib"

TARGET_COL = "target_mora"
ID_COL = "factura_id"
RANDOM_STATE = 42
SELECTION_TOLERANCE_F1 = 0.005

PARAM_GRID = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "class_weight": [None, "balanced"],
    "max_iter": [200, 500],
    "tol": [1e-3, 1e-4],
}


def ensure_dirs() -> None:
    for path in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, ARTIFACTS_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    required = [DATA_PATH, TRAIN_IDS_PATH, TEST_IDS_PATH, MODEL_ARTIFACT_PATH]
    missing = [str(p.relative_to(PROJECT_DIR)) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Faltan entradas requeridas: {missing}")

    df = pd.read_csv(DATA_PATH)
    df[ID_COL] = df[ID_COL].astype(str)
    train_ids = set(pd.read_csv(TRAIN_IDS_PATH)[ID_COL].astype(str))
    test_ids = set(pd.read_csv(TEST_IDS_PATH)[ID_COL].astype(str))
    if train_ids.intersection(test_ids):
        raise ValueError("El split oficial mezcla factura_id entre train y test.")

    artifact = joblib.load(MODEL_ARTIFACT_PATH)
    feature_cols = artifact["feature_cols"]
    missing_cols = [c for c in [ID_COL, TARGET_COL, *feature_cols] if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Columnas faltantes: {missing_cols}")

    train_df = df[df[ID_COL].isin(train_ids)].copy()
    test_df = df[df[ID_COL].isin(test_ids)].copy()
    if train_df.empty or test_df.empty:
        raise ValueError("Train o test oficial quedaron vacios.")
    return df, train_df, test_df, artifact


def encode_targets(train_df: pd.DataFrame, eval_df: pd.DataFrame, artifact: dict) -> tuple[np.ndarray, np.ndarray]:
    encoder = artifact["target_encoder"]
    return (
        encoder.transform(train_df[TARGET_COL].astype(str)),
        encoder.transform(eval_df[TARGET_COL].astype(str)),
    )


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray | None, prefix: str) -> dict:
    row = {
        f"accuracy_{prefix}": accuracy_score(y_true, y_pred),
        f"balanced_accuracy_{prefix}": balanced_accuracy_score(y_true, y_pred),
        f"precision_macro_{prefix}": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"recall_macro_{prefix}": recall_score(y_true, y_pred, average="macro", zero_division=0),
        f"f1_macro_{prefix}": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_proba is not None:
        try:
            row[f"auc_weighted_{prefix}"] = roc_auc_score(
                y_true,
                y_proba,
                labels=list(range(y_proba.shape[1])),
                multi_class="ovr",
                average="weighted",
            )
        except ValueError:
            row[f"auc_weighted_{prefix}"] = np.nan
    else:
        row[f"auc_weighted_{prefix}"] = np.nan
    return row


def select_best_config(df_results: pd.DataFrame, tolerance: float = SELECTION_TOLERANCE_F1) -> pd.Series:
    best_valid = df_results["f1_macro_inner_valid"].max()
    candidates = df_results[
        (df_results["converged"]) & (df_results["f1_macro_inner_valid"] >= best_valid - tolerance)
    ].copy()
    if candidates.empty:
        candidates = df_results.copy()
    return candidates.sort_values(
        ["f1_macro_inner_valid", "balanced_accuracy_inner_valid", "gap_f1_train_valid", "max_iter"],
        ascending=[False, False, True, True],
    ).iloc[0]


def run_experiments(train_df: pd.DataFrame, test_df: pd.DataFrame, artifact: dict) -> pd.DataFrame:
    feature_cols = artifact["feature_cols"]
    preprocessor = clone(artifact["preprocessor"])

    invoice_target = train_df[[ID_COL, TARGET_COL]].drop_duplicates(ID_COL)
    inner_train_ids, inner_valid_ids = train_test_split(
        invoice_target[ID_COL],
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=invoice_target[TARGET_COL],
    )
    inner_train_ids = set(inner_train_ids.astype(str))
    inner_valid_ids = set(inner_valid_ids.astype(str))
    if inner_train_ids.intersection(inner_valid_ids):
        raise ValueError("El split interno mezcla factura_id.")

    inner_train_df = train_df[train_df[ID_COL].isin(inner_train_ids)].copy()
    inner_valid_df = train_df[train_df[ID_COL].isin(inner_valid_ids)].copy()
    y_inner_train = artifact["target_encoder"].transform(inner_train_df[TARGET_COL].astype(str))
    y_inner_valid = artifact["target_encoder"].transform(inner_valid_df[TARGET_COL].astype(str))

    X_inner_train = preprocessor.fit_transform(inner_train_df[feature_cols])
    X_inner_valid = preprocessor.transform(inner_valid_df[feature_cols])

    rows = []
    configs = [dict(zip(PARAM_GRID.keys(), vals)) for vals in itertools.product(*PARAM_GRID.values())]
    for i, params in enumerate(configs, start=1):
        model = LogisticRegression(
            solver="lbfgs",
            random_state=RANDOM_STATE,
            **params,
        )
        start = time.perf_counter()
        model.fit(X_inner_train, y_inner_train)
        elapsed = time.perf_counter() - start

        pred_train = model.predict(X_inner_train)
        pred_valid = model.predict(X_inner_valid)
        proba_train = model.predict_proba(X_inner_train)
        proba_valid = model.predict_proba(X_inner_valid)

        row = {
            "experiment_id": i,
            **params,
            "solver": "lbfgs",
            "fit_time_s": elapsed,
            "n_iter_max": int(np.max(model.n_iter_)),
            "converged": bool(np.max(model.n_iter_) < params["max_iter"]),
        }
        row.update(metric_row(y_inner_train, pred_train, proba_train, "inner_train"))
        row.update(metric_row(y_inner_valid, pred_valid, proba_valid, "inner_valid"))
        row["gap_f1_train_valid"] = row["f1_macro_inner_train"] - row["f1_macro_inner_valid"]
        rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results["class_weight"] = df_results["class_weight"].fillna("none")
    df_results = df_results.sort_values(
        ["f1_macro_inner_valid", "balanced_accuracy_inner_valid", "gap_f1_train_valid"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    df_results["rank_valid"] = np.arange(1, len(df_results) + 1)
    df_results["selected_for_final"] = False
    selected = select_best_config(df_results)
    df_results.loc[df_results["experiment_id"] == selected["experiment_id"], "selected_for_final"] = True

    split_report = {
        "train_facturas_total": int(train_df[ID_COL].nunique()),
        "test_facturas_total": int(test_df[ID_COL].nunique()),
        "inner_train_facturas": int(len(inner_train_ids)),
        "inner_valid_facturas": int(len(inner_valid_ids)),
        "interseccion_inner": int(len(inner_train_ids.intersection(inner_valid_ids))),
        "feature_count": len(feature_cols),
        "target_classes": list(artifact["target_encoder"].classes_),
        "external_test_usage": "solo_comparacion_final_no_usado_para_rankear_configuraciones",
        "selection_rule": f"mejor configuracion convergida dentro de {SELECTION_TOLERANCE_F1:.3f} F1 del mejor score de validacion",
    }
    with open(TABLES_DIR / "reporte_split_interno_validacion.json", "w", encoding="utf-8") as f:
        json.dump(split_report, f, indent=2, ensure_ascii=False)

    return df_results


## 12. Ejecucion de experimentos de hiperparametros

Se entrena cada combinacion con validacion interna por `factura_id`.


In [17]:
df_results = run_experiments(train_df, test_df, artifact)
df_results.to_csv(TABLES_DIR / "experimentos_sensibilidad_hiperparametros.csv", index=False)
df_results.head(10).to_csv(TABLES_DIR / "top10_configuraciones.csv", index=False)
df_results.tail(10).to_csv(TABLES_DIR / "bottom10_configuraciones.csv", index=False)
display(df_results.head(10))


,experiment_id,C,class_weight,max_iter,tol,solver,fit_time_s,n_iter_max,converged,accuracy_inner_train,...,auc_weighted_inner_train,accuracy_inner_valid,balanced_accuracy_inner_valid,precision_macro_inner_valid,recall_macro_inner_valid,f1_macro_inner_valid,auc_weighted_inner_valid,gap_f1_train_valid,rank_valid,selected_for_final
0,26,10.0,none,200,0.0001,lbfgs,0.459252,200,False,0.584075,...,0.826445,0.570776,0.584855,0.563561,0.584855,0.568439,0.817384,0.009193,1,False
1,40,100.0,balanced,500,0.0001,lbfgs,0.410508,236,True,0.583651,...,0.826417,0.566717,0.599245,0.560958,0.599245,0.568293,0.818099,0.015342,2,True
2,22,1.0,balanced,200,0.0001,lbfgs,0.524775,132,True,0.582634,...,0.825910,0.566210,0.599447,0.560863,0.599447,0.568117,0.817381,0.014088,3,False
3,24,1.0,balanced,500,0.0001,lbfgs,0.271484,132,True,0.582634,...,0.825910,0.566210,0.599447,0.560863,0.599447,0.568117,0.817381,0.014088,4,False
4,34,100.0,none,200,0.0001,lbfgs,0.353458,200,False,0.584669,...,0.826399,0.570269,0.584424,0.562948,0.584424,0.567945,0.817294,0.010745,5,False
5,32,10.0,balanced,500,0.0001,lbfgs,0.510922,228,True,0.583143,...,0.826387,0.564942,0.597880,0.559381,0.597880,0.566824,0.818010,0.016206,6,False
6,37,100.0,balanced,200,0.0010,lbfgs,0.216923,55,True,0.580090,...,0.825239,0.565195,0.597731,0.559128,0.597731,0.566779,0.815670,0.012587,7,False
7,39,100.0,balanced,500,0.0010,lbfgs,0.097765,55,True,0.580090,...,0.825239,0.565195,0.597731,0.559128,0.597731,0.566779,0.815670,0.012587,8,False
8,28,10.0,none,500,0.0001,lbfgs,0.431387,204,True,0.583736,...,0.826442,0.569254,0.583056,0.562139,0.583056,0.566700,0.817274,0.010561,9,False
9,29,10.0,balanced,200,0.0010,lbfgs,0.203605,50,True,0.581277,...,0.825103,0.564942,0.598557,0.558582,0.598557,0.566585,0.815434,0.013839,10,False


## 13. Sensibilidad individual, ranking e interacciones

Se analiza cuanto cambia F1 al variar cada hiperparametro y que combinaciones tienen mayor efecto.


In [18]:
def detect_saturation(group: pd.DataFrame, hp_name: str) -> str:
    ordered = group.sort_values(hp_name)
    score_col = "f1_macro_inner_valid" if "f1_macro_inner_valid" in ordered.columns else "f1_macro_mean"
    values = ordered[score_col].to_numpy()
    if len(values) < 3:
        return "sin_evidencia_suficiente"
    best = values.max()
    near_best = ordered.loc[ordered[score_col] >= best - 0.005, hp_name].tolist()
    if len(near_best) >= 2:
        return f"zona_estable_en_{near_best[0]}_a_{near_best[-1]}"
    return "sin_saturacion_clara"


def sensitivity_analysis(df_results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for hp in PARAM_GRID:
        grouped = (
            df_results.groupby(hp, dropna=False)
            .agg(
                f1_macro_mean=("f1_macro_inner_valid", "mean"),
                f1_macro_std=("f1_macro_inner_valid", "std"),
                f1_macro_max=("f1_macro_inner_valid", "max"),
                balanced_accuracy_mean=("balanced_accuracy_inner_valid", "mean"),
                gap_f1_mean=("gap_f1_train_valid", "mean"),
                n_experimentos=("experiment_id", "count"),
            )
            .reset_index()
        )
        best_row = grouped.sort_values(["f1_macro_mean", "balanced_accuracy_mean"], ascending=False).iloc[0]
        spread = grouped["f1_macro_mean"].max() - grouped["f1_macro_mean"].min()
        ordered_scores = grouped["f1_macro_mean"].to_numpy()
        nonlinearity = float(np.std(np.diff(ordered_scores))) if len(ordered_scores) > 2 else 0.0
        saturation = detect_saturation(grouped, hp) if pd.api.types.is_numeric_dtype(grouped[hp]) else "categoria_sin_orden_numerico"
        rows.append(
            {
                "hiperparametro": hp,
                "mejor_valor_promedio": best_row[hp],
                "f1_macro_promedio_mejor_valor": best_row["f1_macro_mean"],
                "rango_f1_macro_promedio": spread,
                "std_promedio": grouped["f1_macro_std"].mean(),
                "no_linealidad_aproximada": nonlinearity,
                "zona_saturacion": saturation,
                "lectura": interpret_hp(hp, spread, nonlinearity, saturation),
            }
        )
    return pd.DataFrame(rows)


def interpret_hp(hp: str, spread: float, nonlinearity: float, saturation: str) -> str:
    if hp == "C":
        return "Controla regularizacion: valores bajos simplifican el modelo; valores altos reducen penalizacion y pueden estabilizarse si no mejoran F1."
    if hp == "class_weight":
        return "Ajusta el peso de clases; es critico si mejora recall macro sin aumentar demasiado el gap train-valid."
    if hp == "max_iter":
        return "Debe leerse como condicion de convergencia; si F1 no cambia al subirlo, hay saturacion computacional."
    if hp == "tol":
        return "Controla criterio de parada; cambios pequenos con mismo F1 indican que el solver ya encontro una zona estable."
    return f"Impacto observado: rango={spread:.4f}, no_linealidad={nonlinearity:.4f}, saturacion={saturation}."


def importance_ranking(df_results: pd.DataFrame) -> pd.DataFrame:
    encoded = pd.get_dummies(df_results[list(PARAM_GRID)].copy(), columns=["class_weight"], drop_first=False)
    y = df_results["f1_macro_inner_valid"].to_numpy()
    rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2)
    rf.fit(encoded, y)
    surrogate_importance = pd.Series(rf.feature_importances_, index=encoded.columns)

    rows = []
    for hp in PARAM_GRID:
        group_means = df_results.groupby(hp)["f1_macro_inner_valid"].mean()
        range_effect = float(group_means.max() - group_means.min())
        if hp == "class_weight":
            spearman = np.nan
            surrogate = float(surrogate_importance[[c for c in surrogate_importance.index if c.startswith("class_weight_")]].sum())
        else:
            spearman = stats.spearmanr(df_results[hp], df_results["f1_macro_inner_valid"]).correlation
            surrogate = float(surrogate_importance.get(hp, 0.0))
        rows.append(
            {
                "hiperparametro": hp,
                "efecto_rango_f1": range_effect,
                "spearman_f1": spearman,
                "importancia_surrogate_rf": surrogate,
                "score_importancia": range_effect * 0.60 + surrogate * 0.40,
            }
        )
    return pd.DataFrame(rows).sort_values("score_importancia", ascending=False).reset_index(drop=True)


def interaction_analysis(df_results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    params = list(PARAM_GRID)
    for hp_a, hp_b in itertools.combinations(params, 2):
        pivot = df_results.pivot_table(
            index=hp_a,
            columns=hp_b,
            values="f1_macro_inner_valid",
            aggfunc="mean",
        )
        best = float(np.nanmax(pivot.to_numpy()))
        worst = float(np.nanmin(pivot.to_numpy()))
        additive_gap = best - worst
        best_position = np.argwhere(pivot.to_numpy() == np.nanmax(pivot.to_numpy()))[0]
        rows.append(
            {
                "hiperparametro_a": hp_a,
                "hiperparametro_b": hp_b,
                "f1_macro_max_interaccion": best,
                "f1_macro_min_interaccion": worst,
                "rango_interaccion": additive_gap,
                "mejor_valor_a": pivot.index[best_position[0]],
                "mejor_valor_b": pivot.columns[best_position[1]],
                "lectura": "Interaccion relevante" if additive_gap >= 0.01 else "Interaccion baja o zona estable",
            }
        )
    return pd.DataFrame(rows).sort_values("rango_interaccion", ascending=False)


In [19]:
sensitivity = sensitivity_analysis(df_results)
ranking = importance_ranking(df_results)
interactions = interaction_analysis(df_results)

sensitivity.to_csv(TABLES_DIR / "sensibilidad_individual_hiperparametros.csv", index=False)
ranking.to_csv(TABLES_DIR / "ranking_importancia_hiperparametros.csv", index=False)
interactions.to_csv(TABLES_DIR / "interacciones_hiperparametros.csv", index=False)

display(sensitivity)
display(ranking)
display(interactions)


,hiperparametro,mejor_valor_promedio,f1_macro_promedio_mejor_valor,rango_f1_macro_promedio,std_promedio,no_linealidad_aproximada,zona_saturacion,lectura
0,C,10.0,0.565893,0.024846,0.003432,0.006805,zona_estable_en_1.0_a_100.0,Controla regularizacion: valores bajos simplif...
1,class_weight,balanced,0.561047,0.004029,0.009891,0.000000,categoria_sin_orden_numerico,Ajusta el peso de clases; es critico si mejora...
2,max_iter,200.0,0.559062,0.000059,0.010344,0.000000,sin_evidencia_suficiente,Debe leerse como condicion de convergencia; si...
3,tol,0.0001,0.560369,0.002674,0.010232,0.000000,sin_evidencia_suficiente,Controla criterio de parada; cambios pequenos ...


,hiperparametro,efecto_rango_f1,spearman_f1,importancia_surrogate_rf,score_importancia
0,C,0.024846,0.796931,0.896297,0.373426
1,class_weight,0.004029,NaN,0.074382,0.032171
2,tol,0.002674,-0.286093,0.026548,0.012223
3,max_iter,0.000059,0.000000,0.002773,0.001145


,hiperparametro_a,hiperparametro_b,f1_macro_max_interaccion,f1_macro_min_interaccion,rango_interaccion,mejor_valor_a,mejor_valor_b,lectura
0,C,class_weight,0.566921,0.534488,0.032433,100.0,balanced,Interaccion relevante
2,C,tol,0.567054,0.540692,0.026362,10.0,0.0001,Interaccion relevante
1,C,max_iter,0.566039,0.541046,0.024992,10.0,200,Interaccion relevante
4,class_weight,tol,0.562147,0.555444,0.006703,balanced,0.0001,Interaccion baja o zona estable
3,class_weight,max_iter,0.561199,0.556807,0.004392,balanced,500,Interaccion baja o zona estable
5,max_iter,tol,0.560428,0.557695,0.002733,200,0.0001,Interaccion baja o zona estable


## 14. Modelo ajustado y comparacion externa

La mejor configuracion por validacion interna se reentrena con todo el train oficial y se compara contra el baseline.


In [20]:

def train_final_best(df_results: pd.DataFrame, train_df: pd.DataFrame, test_df: pd.DataFrame, artifact: dict) -> dict:
    best = select_best_config(df_results)
    params = {k: best[k] for k in PARAM_GRID}
    params["class_weight"] = None if params["class_weight"] == "none" else params["class_weight"]
    params["max_iter"] = int(params["max_iter"])
    params["tol"] = float(params["tol"])
    params["C"] = float(params["C"])

    preprocessor = clone(artifact["preprocessor"])
    X_train = preprocessor.fit_transform(train_df[artifact["feature_cols"]])
    X_test = preprocessor.transform(test_df[artifact["feature_cols"]])
    y_train, y_test = encode_targets(train_df, test_df, artifact)
    model = LogisticRegression(solver="lbfgs", random_state=RANDOM_STATE, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    report = classification_report(
        y_test,
        y_pred,
        labels=list(range(len(artifact["target_encoder"].classes_))),
        target_names=list(artifact["target_encoder"].classes_),
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "clase"})
    report_df.to_csv(TABLES_DIR / "reporte_metricas_por_clase_tuning.csv", index=False)

    cm = confusion_matrix(y_test, y_pred, labels=list(range(len(artifact["target_encoder"].classes_))))
    cm_df = pd.DataFrame(
        cm,
        index=artifact["target_encoder"].classes_,
        columns=artifact["target_encoder"].classes_,
    )
    cm_df.to_csv(TABLES_DIR / "matriz_confusion_mejor_tuning.csv")

    metrics = metric_row(y_test, y_pred, y_proba, "external_test")
    metrics.update(params)
    metrics["selected_experiment_id"] = int(best["experiment_id"])
    metrics["selected_validation_f1_macro"] = float(best["f1_macro_inner_valid"])
    metrics["selected_validation_balanced_accuracy"] = float(best["balanced_accuracy_inner_valid"])
    metrics["selected_inner_converged"] = bool(best["converged"])
    metrics["n_iter_max"] = int(np.max(model.n_iter_))
    metrics["converged_external_fit"] = bool(np.max(model.n_iter_) < params["max_iter"])
    metrics["selection_rule"] = f"mejor configuracion convergida dentro de {SELECTION_TOLERANCE_F1:.3f} F1 del mejor score de validacion"
    with open(TABLES_DIR / "mejor_configuracion_tuning.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    joblib.dump(
        {
            "model_name": "Logistic Regression tuned",
            "model": model,
            "preprocessor": preprocessor,
            "feature_cols": artifact["feature_cols"],
            "target_encoder": artifact["target_encoder"],
            "params": params,
            "selection_rule": metrics["selection_rule"],
        },
        ARTIFACTS_DIR / "best_tuned_logistic_regression.joblib",
    )
    return metrics


In [21]:
final_metrics = train_final_best(df_results, train_df, test_df, artifact)
comparison = pd.DataFrame([
    {"modelo": "baseline_fase4", "f1_macro_test": baseline["f1_macro"], "balanced_accuracy_test": baseline["balanced_accuracy"]},
    {"modelo": "tuning_fase5", "f1_macro_test": final_metrics["f1_macro_external_test"], "balanced_accuracy_test": final_metrics["balanced_accuracy_external_test"]},
])
display(comparison)
display(pd.read_csv(TABLES_DIR / "reporte_metricas_por_clase_tuning.csv"))


,modelo,f1_macro_test,balanced_accuracy_test
0,baseline_fase4,0.570000,0.600614
1,tuning_fase5,0.569655,0.599260


,clase,precision,recall,f1-score,support
0,+30,0.418507,0.529960,0.467685,751.000000
1,+60,0.522946,0.403957,0.455814,1213.000000
2,+90,0.690741,0.573405,0.626627,1301.000000
3,on_time,0.616736,0.889717,0.728493,671.000000
4,accuracy,0.566819,0.566819,0.566819,0.566819
5,macro avg,0.562232,0.599260,0.569655,3936.000000
6,weighted avg,0.574470,0.566819,0.561025,3936.000000


## 15. Graficos y cierre de hiperparametros

Se exportan graficos de ranking, sensibilidad e interacciones. Al final se registra un resumen compacto de la fase.


In [22]:
def make_figures(
    df_results: pd.DataFrame,
    sensitivity: pd.DataFrame,
    ranking: pd.DataFrame,
    interactions: pd.DataFrame,
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    top = df_results.sort_values("rank_valid").head(15).iloc[::-1]
    ax.barh(top["experiment_id"].astype(str), top["f1_macro_inner_valid"])
    ax.set_title("Top 15 configuraciones por F1-macro validacion")
    ax.set_xlabel("F1-macro validacion")
    ax.set_ylabel("Experimento")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "ranking_f1_macro_experimentos.png", dpi=150)
    plt.close(fig)

    for hp in PARAM_GRID:
        group = df_results.groupby(hp)["f1_macro_inner_valid"].agg(["mean", "std"]).reset_index()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(group[hp].astype(str), group["mean"], yerr=group["std"].fillna(0), marker="o")
        ax.set_title(f"Sensibilidad individual: {hp}")
        ax.set_ylabel("F1-macro validacion")
        ax.set_xlabel(hp)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"sensibilidad_{hp}.png", dpi=150)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ordered = ranking.iloc[::-1]
    ax.barh(ordered["hiperparametro"], ordered["score_importancia"])
    ax.set_title("Ranking de importancia de hiperparametros")
    ax.set_xlabel("Score combinado")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "ranking_importancia_hiperparametros.png", dpi=150)
    plt.close(fig)

    for _, row in interactions.head(4).iterrows():
        hp_a, hp_b = row["hiperparametro_a"], row["hiperparametro_b"]
        pivot = df_results.pivot_table(index=hp_a, columns=hp_b, values="f1_macro_inner_valid", aggfunc="mean")
        fig, ax = plt.subplots(figsize=(7, 5))
        im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="viridis")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([str(c) for c in pivot.columns], rotation=30, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([str(i) for i in pivot.index])
        ax.set_xlabel(hp_b)
        ax.set_ylabel(hp_a)
        ax.set_title(f"Interaccion {hp_a} x {hp_b}")
        fig.colorbar(im, ax=ax, label="F1-macro validacion")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"interaccion_{hp_a}_x_{hp_b}.png", dpi=150)
        plt.close(fig)


In [23]:

make_figures(df_results, sensitivity, ranking, interactions)

selected_config = df_results[df_results["selected_for_final"]].iloc[0]
recommendations = pd.DataFrame([
    {
        "decision": "mantener_modelo_base_como_referencia",
        "criterio": "la configuracion ajustada no mejora claramente el test externo",
        "valor": f"baseline={baseline['f1_macro']:.4f}; tuning={final_metrics['f1_macro_external_test']:.4f}",
    },
    {
        "decision": "priorizar_C_en_busquedas_futuras",
        "criterio": "mayor score de importancia e interacciones mas fuertes",
        "valor": ranking.iloc[0]["hiperparametro"],
    },
    {
        "decision": "preferir_configuraciones_convergidas",
        "criterio": "la diferencia de validacion frente al maximo fue menor que la tolerancia definida",
        "valor": f"experimento_seleccionado={int(selected_config['experiment_id'])}",
    },
    {
        "decision": "monitorear_cartera_critica",
        "criterio": "mayor degradacion observada en escenarios de drift",
        "valor": "drift_cartera_critica_q75",
    },
])
recommendations.to_csv(TABLES_DIR / "recomendaciones_tuning_hiperparametros.csv", index=False)

summary = {
    "baseline_f1_macro": float(baseline["f1_macro"]),
    "worst_noise_scenario": noise_summary.sort_values("degradacion_f1_macro_pct_mean", ascending=False).iloc[0]["escenario"],
    "worst_drift_scenario": drift_results.sort_values("degradacion_f1_macro_pct", ascending=False).iloc[0]["escenario"],
    "n_hyperparameter_experiments": int(len(df_results)),
    "best_validation_f1_macro": float(df_results.iloc[0]["f1_macro_inner_valid"]),
    "selected_validation_f1_macro": float(selected_config["f1_macro_inner_valid"]),
    "selected_experiment_id": int(selected_config["experiment_id"]),
    "selected_converged": bool(selected_config["converged"]),
    "tuned_external_f1_macro": float(final_metrics["f1_macro_external_test"]),
    "most_important_hyperparameter": ranking.iloc[0]["hiperparametro"],
}
with open(TABLES_DIR / "resumen_sensibilidad_hiperparametros.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
with open(ARTIFACTS_DIR / "scenario_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "source_dataset": str(DATA_PATH.relative_to(PROJECT_DIR)),
        "source_model_artifact": str(MODEL_ARTIFACT_PATH.relative_to(PROJECT_DIR)),
        "feature_count": len(feature_cols),
        "seeds": RANDOM_SEEDS,
        "noise_levels": [0.05, 0.10, 0.20],
        "missing_levels": [0.05, 0.15, 0.30],
        "psi_thresholds": {"moderate": 0.10, "high": 0.25},
    }, f, indent=2, ensure_ascii=False)

risk_matrix = pd.DataFrame(
    [
        ["Missing dirigido gestion", "Alta", "F1-macro", "Alerta de caida CRM y revision humana"],
        ["Drift temporal futuro", "Alta", "Recall +90", "Monitorear PSI/KS mensual y reentrenar por ventana"],
        ["Drift cartera critica Q75", "Alta", "F1-macro y recall", "Politicas especificas para cartera concentrada"],
        ["Inputs extremos", "Critica", "Estabilidad tecnica", "Validadores de contrato antes de inferencia"],
        ["Volumen 100x", "Media", "Latencia", "Batch asincrono y medicion p95"],
        ["Columna requerida faltante", "Critica", "Fallo de schema", "Validacion estricta de schema"],
    ],
    columns=["escenario", "severidad", "metrica_afectada", "mitigacion"],
)
risk_matrix.to_csv(TABLES_DIR / "risk_matrix.csv", index=False)

output_counts = pd.DataFrame([
    {"tipo": "tablas", "cantidad": len(list(TABLES_DIR.glob("*")))},
    {"tipo": "figuras", "cantidad": len(list(FIGURES_DIR.glob("*")))},
    {"tipo": "artefactos", "cantidad": len(list(ARTIFACTS_DIR.glob("*")))},
])
display(recommendations)
display(pd.Series(summary).to_frame("valor"))
display(output_counts)


,decision,criterio,valor
0,mantener_modelo_base_como_referencia,la configuracion ajustada no mejora claramente...,baseline=0.5700; tuning=0.5697
1,priorizar_C_en_busquedas_futuras,mayor score de importancia e interacciones mas...,C
2,preferir_configuraciones_convergidas,la diferencia de validacion frente al maximo f...,experimento_seleccionado=40
3,monitorear_cartera_critica,mayor degradacion observada en escenarios de d...,drift_cartera_critica_q75


,valor
baseline_f1_macro,0.57
worst_noise_scenario,outlier_critico_combinado
worst_drift_scenario,drift_cartera_critica_q75
n_hyperparameter_experiments,40
best_validation_f1_macro,0.568439
selected_validation_f1_macro,0.568293
selected_experiment_id,40
selected_converged,True
tuned_external_f1_macro,0.569655
most_important_hyperparameter,C


,tipo,cantidad
0,tablas,24
1,figuras,15
2,artefactos,2


## 16. Componente 2: sensibilidad de clustering

El clustering no tiene `target`, por lo que no se optimiza por F1. Se revisan hiperparametros de segmentacion con metricas internas y estabilidad operativa.


In [24]:

from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import RobustScaler, StandardScaler

CLIENT_CLUSTER_BASE_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "client_features_clustering_base.csv"
CLIENT_CLUSTER_FEATURES_PATH = PROJECT_DIR / "03_preparacion" / "outputs" / "client_clustering_features_selected.csv"
CLUSTER_CLIENT_COL = "cliente_id"
LOG1P_BASE_VARS = [
    "monto",
    "monto_promedio_hist",
    "ratio_monto",
    "promesas_total",
    "num_promesas_rotas",
    "num_gestiones_factura",
    "dias_mora_observable",
    "num_no_contesta_cons",
    "dias_hasta_vence_pos",
    "dias_desde_ultima_gestion",
]
KMEANS_CLUSTER_GRID = {
    "n_clusters": [2, 3, 4, 5, 6],
    "init": ["k-means++", "random"],
    "n_init": [10, 20, 50],
    "scaler": ["robust", "standard"],
    "use_log1p": [True, False],
}
DBSCAN_CLUSTER_GRID = {
    "eps": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5],
    "min_samples": [3, 5, 10],
    "scaler": ["robust"],
    "use_log1p": [True],
}


def needs_log1p_cluster(column: str) -> bool:
    return any(column.startswith(f"{base}_") for base in LOG1P_BASE_VARS)


def load_clustering_inputs() -> tuple[pd.DataFrame, list[str]]:
    required = [CLIENT_CLUSTER_BASE_PATH, CLIENT_CLUSTER_FEATURES_PATH]
    missing = [str(p.relative_to(PROJECT_DIR)) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Faltan entradas de clustering: {missing}")
    client_df = pd.read_csv(CLIENT_CLUSTER_BASE_PATH)
    feature_df = pd.read_csv(CLIENT_CLUSTER_FEATURES_PATH)
    feature_col = "feature" if "feature" in feature_df.columns else feature_df.columns[0]
    features = feature_df[feature_col].astype(str).tolist()
    missing_cols = [c for c in [CLUSTER_CLIENT_COL, *features] if c not in client_df.columns]
    if missing_cols:
        raise ValueError(f"Columnas faltantes en base de clustering: {missing_cols}")
    if client_df[CLUSTER_CLIENT_COL].duplicated().any():
        raise ValueError("La base de clustering debe tener una fila por cliente_id.")
    return client_df, features


def prepare_cluster_matrix(client_df: pd.DataFrame, features: list[str], scaler: str, use_log1p: bool) -> np.ndarray:
    X = client_df[features].copy()
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        if use_log1p and needs_log1p_cluster(col):
            X[col] = np.log1p(np.maximum(X[col].fillna(0), 0))
        elif X[col].isna().any():
            X[col] = X[col].fillna(X[col].median())
    zero_var_cols = [col for col in X.columns if X[col].nunique(dropna=True) <= 1]
    if zero_var_cols:
        X = X.drop(columns=zero_var_cols)
    if scaler == "robust":
        return RobustScaler().fit_transform(X)
    if scaler == "standard":
        return StandardScaler().fit_transform(X)
    return X.to_numpy()


def cluster_quality(X: np.ndarray, labels: np.ndarray, method: str) -> dict:
    labels = np.asarray(labels)
    noise_mask = labels == -1
    valid_mask = ~noise_mask
    valid_labels = labels[valid_mask]
    n_clusters = len(set(valid_labels))
    counts = pd.Series(valid_labels).value_counts().sort_index()
    if n_clusters > 1 and valid_mask.sum() > n_clusters:
        sil = float(silhouette_score(X[valid_mask], valid_labels))
        dbi = float(davies_bouldin_score(X[valid_mask], valid_labels))
        chi = float(calinski_harabasz_score(X[valid_mask], valid_labels))
    else:
        sil, dbi, chi = np.nan, np.nan, np.nan
    min_share = float(counts.min() / len(labels)) if not counts.empty else 0.0
    max_share = float(counts.max() / len(labels)) if not counts.empty else 0.0
    return {
        "method": method,
        "n_clusters_found": int(n_clusters),
        "silhouette": sil,
        "davies_bouldin": dbi,
        "calinski_harabasz": chi,
        "noise_ratio": float(noise_mask.mean()),
        "min_cluster_share": min_share,
        "max_cluster_share": max_share,
        "n_noise": int(noise_mask.sum()),
        "cluster_size_distribution": ";".join(f"{idx}:{int(val)}" for idx, val in counts.items()),
    }


def run_clustering_hyperparameter_experiments(client_df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    matrix_cache = {}
    for scaler in KMEANS_CLUSTER_GRID["scaler"]:
        for use_log1p in KMEANS_CLUSTER_GRID["use_log1p"]:
            matrix_cache[(scaler, use_log1p)] = prepare_cluster_matrix(client_df, features, scaler, use_log1p)

    for values in itertools.product(*KMEANS_CLUSTER_GRID.values()):
        params = dict(zip(KMEANS_CLUSTER_GRID.keys(), values))
        X = matrix_cache[(params["scaler"], params["use_log1p"])]
        model = KMeans(
            n_clusters=params["n_clusters"],
            init=params["init"],
            n_init=params["n_init"],
            random_state=RANDOM_STATE,
        )
        labels = model.fit_predict(X)
        row = {"algorithm": "KMeans", **params, "eps": np.nan, "min_samples": np.nan, "inertia": float(model.inertia_)}
        row.update(cluster_quality(X, labels, "KMeans"))
        rows.append(row)

    for values in itertools.product(*DBSCAN_CLUSTER_GRID.values()):
        params = dict(zip(DBSCAN_CLUSTER_GRID.keys(), values))
        X = matrix_cache.get((params["scaler"], params["use_log1p"]))
        if X is None:
            X = prepare_cluster_matrix(client_df, features, params["scaler"], params["use_log1p"])
            matrix_cache[(params["scaler"], params["use_log1p"])] = X
        labels = DBSCAN(eps=params["eps"], min_samples=params["min_samples"]).fit_predict(X)
        row = {
            "algorithm": "DBSCAN",
            "n_clusters": np.nan,
            "init": np.nan,
            "n_init": np.nan,
            **params,
            "inertia": np.nan,
        }
        row.update(cluster_quality(X, labels, "DBSCAN"))
        rows.append(row)
    return pd.DataFrame(rows)


def clustering_sensitivity(df_cluster: pd.DataFrame) -> pd.DataFrame:
    rows = []
    param_cols = ["algorithm", "n_clusters", "init", "n_init", "scaler", "use_log1p", "eps", "min_samples"]
    for param in param_cols:
        valid = df_cluster[df_cluster[param].notna()].copy()
        if valid[param].nunique(dropna=True) < 2:
            continue
        grouped = valid.groupby(param, dropna=False).agg(
            silhouette_mean=("silhouette", "mean"),
            silhouette_max=("silhouette", "max"),
            davies_bouldin_mean=("davies_bouldin", "mean"),
            noise_ratio_mean=("noise_ratio", "mean"),
            min_cluster_share_mean=("min_cluster_share", "mean"),
            n_experiments=("method", "count"),
        ).reset_index()
        rows.append({
            "hiperparametro": param,
            "mejor_valor_promedio": grouped.sort_values(["silhouette_mean", "min_cluster_share_mean"], ascending=False).iloc[0][param],
            "rango_silhouette_promedio": float(grouped["silhouette_mean"].max() - grouped["silhouette_mean"].min()),
            "rango_noise_ratio_promedio": float(grouped["noise_ratio_mean"].max() - grouped["noise_ratio_mean"].min()),
            "n_valores": int(grouped[param].nunique(dropna=True)),
            "lectura": "Mayor rango implica mayor sensibilidad del componente de clustering a este parametro.",
        })
    return pd.DataFrame(rows).sort_values("rango_silhouette_promedio", ascending=False)


def clustering_interactions(df_cluster: pd.DataFrame) -> pd.DataFrame:
    rows = []
    km = df_cluster[df_cluster["algorithm"] == "KMeans"].copy()
    params = ["n_clusters", "init", "n_init", "scaler", "use_log1p"]
    for a, b in itertools.combinations(params, 2):
        pivot = km.pivot_table(index=a, columns=b, values="silhouette", aggfunc="mean")
        if pivot.empty:
            continue
        values = pivot.to_numpy(dtype=float)
        rows.append({
            "hiperparametro_a": a,
            "hiperparametro_b": b,
            "silhouette_max_interaccion": float(np.nanmax(values)),
            "silhouette_min_interaccion": float(np.nanmin(values)),
            "rango_interaccion": float(np.nanmax(values) - np.nanmin(values)),
            "lectura": "Interaccion relevante" if float(np.nanmax(values) - np.nanmin(values)) >= 0.05 else "Interaccion baja o estable",
        })
    return pd.DataFrame(rows).sort_values("rango_interaccion", ascending=False)


def make_clustering_figures(df_cluster: pd.DataFrame, sensitivity_df: pd.DataFrame) -> None:
    km = df_cluster[df_cluster["algorithm"] == "KMeans"].copy()
    fig, ax = plt.subplots(figsize=(8, 4))
    km.groupby("n_clusters")["silhouette"].mean().plot(marker="o", ax=ax)
    ax.set_title("Sensibilidad KMeans por numero de clusters")
    ax.set_xlabel("n_clusters")
    ax.set_ylabel("Silhouette promedio")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "clustering_sensibilidad_kmeans_k.png", dpi=150)
    plt.close(fig)

    if not sensitivity_df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ordered = sensitivity_df.sort_values("rango_silhouette_promedio").tail(8)
        ax.barh(ordered["hiperparametro"], ordered["rango_silhouette_promedio"])
        ax.set_title("Importancia de hiperparametros de clustering")
        ax.set_xlabel("Rango de silhouette promedio")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "clustering_ranking_hiperparametros.png", dpi=150)
        plt.close(fig)


client_cluster_df, cluster_features = load_clustering_inputs()
cluster_results = run_clustering_hyperparameter_experiments(client_cluster_df, cluster_features)
cluster_results.to_csv(TABLES_DIR / "clustering_hyperparameter_experiments.csv", index=False)

cluster_sensitivity = clustering_sensitivity(cluster_results)
cluster_sensitivity.to_csv(TABLES_DIR / "clustering_sensibilidad_hiperparametros.csv", index=False)
cluster_ranking = cluster_sensitivity.rename(columns={"rango_silhouette_promedio": "score_importancia"})[
    ["hiperparametro", "score_importancia", "mejor_valor_promedio", "lectura"]
]
cluster_ranking.to_csv(TABLES_DIR / "clustering_ranking_importancia_hiperparametros.csv", index=False)
cluster_interactions = clustering_interactions(cluster_results)
cluster_interactions.to_csv(TABLES_DIR / "clustering_interacciones_hiperparametros.csv", index=False)

make_clustering_figures(cluster_results, cluster_sensitivity)

cluster_decision = pd.DataFrame([
    {
        "decision": "mantener_kmeans_k3_como_segmentacion_principal",
        "criterio": "k=2 maximiza silhouette pero separa casi toda la base en un grupo; k=3 conserva lectura operativa ya documentada en fase 4",
        "valor": "clustering_metrics.csv fase 4 y sensibilidad fase 5",
    },
    {
        "decision": "no_tratar_clustering_como_prediccion",
        "criterio": "no hay target ni F1; se evalua cohesion, separacion, ruido y balance de tamanos",
        "valor": "silhouette, davies_bouldin, calinski_harabasz, noise_ratio, min_cluster_share",
    },
    {
        "decision": "descartar_dbscan_como_modelo_principal",
        "criterio": "la sensibilidad a eps/min_samples genera mucho ruido o clusters insuficientes en la mayoria de configuraciones",
        "valor": f"noise_ratio_promedio={cluster_results[cluster_results['algorithm'] == 'DBSCAN']['noise_ratio'].mean():.2f}",
    },
])
cluster_decision.to_csv(TABLES_DIR / "clustering_recomendaciones_hiperparametros.csv", index=False)

display(cluster_results.sort_values(["algorithm", "silhouette"], ascending=[True, False]).head(12))
display(cluster_sensitivity)
display(cluster_interactions.head(10))
display(cluster_decision)


,algorithm,n_clusters,init,n_init,scaler,use_log1p,eps,min_samples,inertia,method,n_clusters_found,silhouette,davies_bouldin,calinski_harabasz,noise_ratio,min_cluster_share,max_cluster_share,n_noise,cluster_size_distribution
136,DBSCAN,NaN,NaN,NaN,robust,True,3.0,5.0,NaN,DBSCAN,2,0.295612,0.989605,23.465981,0.525,0.045,0.430,105,0:86;1:9
135,DBSCAN,NaN,NaN,NaN,robust,True,3.0,3.0,NaN,DBSCAN,2,0.138141,1.314536,5.352583,0.470,0.020,0.510,94,0:102;1:4
132,DBSCAN,NaN,NaN,NaN,robust,True,2.5,3.0,NaN,DBSCAN,7,0.109453,1.221157,7.774419,0.720,0.015,0.175,144,0:35;1:3;2:3;3:5;4:4;5:3;6:3
120,DBSCAN,NaN,NaN,NaN,robust,True,0.5,3.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
121,DBSCAN,NaN,NaN,NaN,robust,True,0.5,5.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
122,DBSCAN,NaN,NaN,NaN,robust,True,0.5,10.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
123,DBSCAN,NaN,NaN,NaN,robust,True,1.0,3.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
124,DBSCAN,NaN,NaN,NaN,robust,True,1.0,5.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
125,DBSCAN,NaN,NaN,NaN,robust,True,1.0,10.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,
126,DBSCAN,NaN,NaN,NaN,robust,True,1.5,3.0,NaN,DBSCAN,0,NaN,NaN,NaN,1.000,0.000,0.000,200,


,hiperparametro,mejor_valor_promedio,rango_silhouette_promedio,rango_noise_ratio_promedio,n_valores,lectura
1,n_clusters,2.0,0.361953,0.000000,5,Mayor rango implica mayor sensibilidad del com...
4,scaler,robust,0.209308,0.213827,2,Mayor rango implica mayor sensibilidad del com...
7,min_samples,5.0,0.171815,0.084286,3,Mayor rango implica mayor sensibilidad del com...
0,algorithm,KMeans,0.121686,0.824762,2,Mayor rango implica mayor sensibilidad del com...
6,eps,3.0,0.107424,0.646667,7,Mayor rango implica mayor sensibilidad del com...
5,use_log1p,False,0.014767,0.213827,2,Mayor rango implica mayor sensibilidad del com...
2,init,k-means++,0.001856,0.000000,2,Mayor rango implica mayor sensibilidad del com...
3,n_init,20.0,0.001260,0.000000,3,Mayor rango implica mayor sensibilidad del com...


,hiperparametro_a,hiperparametro_b,silhouette_max_interaccion,silhouette_min_interaccion,rango_interaccion,lectura
2,n_clusters,scaler,0.826818,0.162115,0.664703,Interaccion relevante
3,n_clusters,use_log1p,0.546536,0.168332,0.378204,Interaccion relevante
1,n_clusters,n_init,0.546498,0.182731,0.363767,Interaccion relevante
0,n_clusters,init,0.546578,0.183212,0.363367,Interaccion relevante
9,scaler,use_log1p,0.413430,0.182888,0.230542,Interaccion relevante
5,init,scaler,0.413094,0.190889,0.222205,Interaccion relevante
7,n_init,scaler,0.413784,0.191899,0.221885,Interaccion relevante
6,init,use_log1p,0.308100,0.297052,0.011047,Interaccion baja o estable
8,n_init,use_log1p,0.307840,0.297296,0.010544,Interaccion baja o estable
4,init,n_init,0.304384,0.300378,0.004006,Interaccion baja o estable


,decision,criterio,valor
0,mantener_kmeans_k3_como_segmentacion_principal,k=2 maximiza silhouette pero separa casi toda ...,clustering_metrics.csv fase 4 y sensibilidad f...
1,no_tratar_clustering_como_prediccion,"no hay target ni F1; se evalua cohesion, separ...","silhouette, davies_bouldin, calinski_harabasz,..."
2,descartar_dbscan_como_modelo_principal,la sensibilidad a eps/min_samples genera mucho...,noise_ratio_promedio=0.82
